In [1]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list

from __future__ import annotations

import os  # needed navigate the system to get the input data

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

### initiate extractor

In [2]:
# Instantiate the extractor
paramPath = '/host/d/Github/Osteosarcoma/radiomics_settings/MR_setting_image.yaml'
extractor = featureextractor.RadiomicsFeatureExtractor(paramPath)

print('Extraction parameters:\n\t', extractor.settings)
print('Enabled filters:\n\t', extractor.enabledImagetypes)
print('Enabled features:\n\t', extractor.enabledFeatures)

Extraction parameters:
	 {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': True, 'normalizeScale': 100, 'removeOutliers': None, 'resampledPixelSpacing': [1, 1, 1], 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 25, 'voxelArrayShift': 300, 'geometryTolerance': 0.0001}
Enabled filters:
	 {'Original': {}, 'LoG': {'sigma': [2.0, 4.0]}, 'Wavelet': {}}
Enabled features:
	 {'shape': None, 'firstorder': None, 'glcm': ['Autocorrelation', 'JointAverage', 'ClusterProminence', 'ClusterShade', 'ClusterTendency', 'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy', 'DifferenceVariance', 'JointEnergy', 'JointEntropy', 'Imc1', 'Imc2', 'Idm', 'Idmn', 'Id', 'Idn', 'InverseVariance', 'MaximumProbability', 'SumEntropy', 'SumSquares'], 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None}


### define patient list

In [3]:
patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx'
# patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/cases_with_label_reader2.xlsx'
build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')
# show one example of the image and mask
print('patient set:', patient_set_list[0], 'patient index:', patient_index_list[0], 'label:', label_list[0], 'image path:', image_path_list[0], 'mask path:', mask_path_list[0])


Number of cases to process: 351
patient set: set_1 patient index: 1 label: 0 image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz mask path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz


### extract features

In [4]:
# ============================================================
# Extract whole-image radiomics features
# Supports set1 + set2 + set3 and resumes from existing table
# ============================================================

out_excel = '/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements.xlsx'
os.makedirs(os.path.dirname(out_excel), exist_ok=True)

# If True: rerun every case and rebuild radiomics_measurements.xlsx from scratch.
# If False: read existing radiomics_measurements.xlsx and skip cases already present.
overwrite_existing_measurements = False

id_cols = ["Patient_set", "Patient_index"]
front_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]


def normalize_case_key(patient_set, patient_index):
    """Make patient identifiers stable across Excel numeric/string reads."""
    ps = str(patient_set).strip()
    pi = str(patient_index).strip()
    try:
        f = float(pi)
        if f.is_integer():
            pi = str(int(f))
    except Exception:
        pass
    return ps, pi


# ============================================================
# 1. Load existing measurements unless overwrite is requested
# ============================================================

if (not overwrite_existing_measurements) and os.path.isfile(out_excel):
    existing_df = pd.read_excel(out_excel, dtype={"Patient_index": str})

    if all(col in existing_df.columns for col in id_cols):
        existing_df["Patient_set"] = existing_df["Patient_set"].astype(str).str.strip()
        existing_df["Patient_index"] = existing_df["Patient_index"].apply(lambda x: normalize_case_key("", x)[1])

        # Drop duplicate old rows by case key to avoid accumulating duplicated measurements.
        before_n = existing_df.shape[0]
        existing_df = existing_df.drop_duplicates(subset=id_cols, keep="first").reset_index(drop=True)
        after_n = existing_df.shape[0]
        if after_n < before_n:
            print(f"Dropped duplicate existing rows: {before_n - after_n}")
    else:
        raise KeyError(f"Existing measurement table must contain columns {id_cols}: {out_excel}")

    rows = existing_df.to_dict(orient="records")
    completed_keys = set(
        normalize_case_key(row["Patient_set"], row["Patient_index"])
        for _, row in existing_df[id_cols].iterrows()
    )

    print("Loaded existing measurements:", out_excel)
    print("Existing measured cases:", len(completed_keys))
else:
    existing_df = pd.DataFrame()
    rows = []
    completed_keys = set()
    if overwrite_existing_measurements:
        print("overwrite_existing_measurements=True: all cases will be rerun.")
    else:
        print("No existing measurement table found. All cases will be run.")


# ============================================================
# 2. Run missing cases
# ============================================================

new_success_count = 0
skip_existing_count = 0
skip_missing_file_count = 0
skip_failed_count = 0

for i in range(0, len(patient_index_list)):
    img_p = image_path_list[i]
    msk_p = mask_path_list[i]
    cid = patient_index_list[i]
    patient_set = patient_set_list[i]
    case_key = normalize_case_key(patient_set, cid)

    print('\n============================================================')
    print('i', i, 'patient set:', patient_set, 'patient index:', cid)
    print('  image path:', img_p)
    print('  mask path :', msk_p)

    if (not overwrite_existing_measurements) and case_key in completed_keys:
        skip_existing_count += 1
        print('  [skip] existing measurement found in radiomics_measurements.xlsx')
        continue

    if (not os.path.isfile(img_p)) or (not os.path.isfile(msk_p)):
        skip_missing_file_count += 1
        print('  [skip] missing file')
        continue

    try:
        result = extractor.execute(img_p, msk_p)
    except Exception as e:
        skip_failed_count += 1
        print(f'  [skip] extractor failed: {e}')
        continue

    # Keep only radiomics features (drop diagnostics)
    feats = {k: v for k, v in result.items() if not k.startswith("diagnostics_")}
    feats["Patient_set"] = case_key[0]
    feats["Patient_index"] = case_key[1]
    feats["Image_filepath"] = img_p
    feats["Mask_filepath"] = msk_p

    rows.append(feats)
    completed_keys.add(case_key)
    new_success_count += 1

    # Build dataframe and save after each successful case so interruption will not lose extracted data.
    df = pd.DataFrame(rows)

    # Keep identifiers first, then all radiomics features.
    existing_front_cols = [c for c in front_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in existing_front_cols]
    df = df[existing_front_cols + other_cols]

    # Stable ordering by set/index where possible.
    df["_patient_index_sort"] = pd.to_numeric(df["Patient_index"], errors="coerce")
    df = (
        df
        .sort_values(by=["Patient_set", "_patient_index_sort", "Patient_index"], kind="mergesort")
        .drop(columns=["_patient_index_sort"])
        .reset_index(drop=True)
    )

    df.to_excel(out_excel, index=False)
    print(f'  [saved] total measured cases: {len(df)} -> {out_excel}')


# ============================================================
# 3. Final save and summary
# ============================================================

if len(rows) > 0:
    df = pd.DataFrame(rows)
    existing_front_cols = [c for c in front_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in existing_front_cols]
    df = df[existing_front_cols + other_cols]
    df["_patient_index_sort"] = pd.to_numeric(df["Patient_index"], errors="coerce")
    df = (
        df
        .drop_duplicates(subset=id_cols, keep="first")
        .sort_values(by=["Patient_set", "_patient_index_sort", "Patient_index"], kind="mergesort")
        .drop(columns=["_patient_index_sort"])
        .reset_index(drop=True)
    )
    df.to_excel(out_excel, index=False)
else:
    df = pd.DataFrame()

print('\n============================================================')
print('Extraction loop finished.')
print('Output:', out_excel)
print('overwrite_existing_measurements:', overwrite_existing_measurements)
print('New successful cases:', new_success_count)
print('Skipped existing cases:', skip_existing_count)
print('Skipped missing files:', skip_missing_file_count)
print('Skipped extractor failures:', skip_failed_count)
print('Final measured cases:', df.shape[0])
if df.shape[0] > 0 and 'Patient_set' in df.columns:
    print('\nFinal measured cases by Patient_set:')
    print(df['Patient_set'].value_counts().sort_index().to_string())

Loaded existing measurements: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements.xlsx
Existing measured cases: 330

i 0 patient set: set_1 patient index: 1
  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz
  mask path : /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz
  [skip] existing measurement found in radiomics_measurements.xlsx

i 1 patient set: set_1 patient index: 5
  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/5/img.nii.gz
  mask path : /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/5/label.nii.gz
  [skip] existing measurement found in radiomics_measurements.xlsx

i 2 patient set: set_1 patient index: 7
  image path: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/7/img.nii.gz
  mask path : /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/7/label.nii.gz
  [skip] existing measurement found in radiomics_measurements.xlsx

i 3 patient set: set_1 patient index: 8
  image 

### normalize features

#### normalize for reader 1

In [5]:
### normalize features
df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements.xlsx')
### normalize features to [0,1] 
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]
df_features = df[feature_cols]
df_features_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=feature_cols)
df_scaled = pd.concat([df[non_feature_cols], df_features_scaled], axis=1)
df_scaled.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx', index=False)

In [6]:
feature_min = scaler.data_min_
feature_max = scaler.data_max_

# 通过radiomics_measurements_normalized.xlsx计算得到的feature_min和feature_max更新radiomics_features_list.xlsx中的对应列, 然后每一行是一个feature, 这样就知道每个feature的min和max值了
df_scaled = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
# 我首先需要通过df_scaled的columns来获取feature list
feature_list = df_scaled.columns.tolist()
feature_list = [f for f in feature_list if f not in non_feature_cols]
feature_table = pd.DataFrame({'feature_name': feature_list, 'feature_min': feature_min, 'feature_max': feature_max})
feature_table.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_features_list.xlsx', index=False)

#### normalize for reader 2, will use data_min and data_max from reader 1 normalization

In [25]:
# for each feature, get the scaler.data_min_ and data_max_ from radiomics_features_list.xlsx, and then use it for normalizaton
scale_df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_features_list.xlsx')
df_reader2 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_reader2.xlsx')
feature_names = [c for c in df_reader2.columns if c not in non_feature_cols]
for feature in feature_names:
    f_min = scale_df.loc[scale_df['feature_name'] == feature, 'feature_min'].values[0]
    f_max = scale_df.loc[scale_df['feature_name'] == feature, 'feature_max'].values[0]
    df_reader2[feature] = (df_reader2[feature] - f_min) / (f_max - f_min)
df_reader2.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized_reader2.xlsx', index=False)

